In [0]:
# Databricks notebook source
# DBTITLE 1,Initialize and Define Ingestion Paths
from pyspark.sql import functions as F
from pyspark.sql.types import StructType
import sys

# Define landing parameters
source_csv_path = "/Volumes/workspace/bronze/v_gii_raw_landing/18100245.csv"
target_table_name = "workspace.bronze.grocery_prices"

# DBTITLE 2,Ingest Raw CSV to Dataframe
try:
    print(f"Starting ingestion from: {source_csv_path}")
    
    # Read everything as string initially to maintain a robust landing step (Schema-on-Read mitigation)
    raw_df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "false") 
              .load(source_csv_path))
              
    if raw_df.isEmpty():
        raise ValueError("Source CSV file is empty.")

    # DBTITLE 3,Add Lineage Metadata
    bronze_df = (raw_df
                 .withColumn("ingestion_timestamp", F.current_timestamp())
                 .withColumn("source_file_name", F.lit(source_csv_path.split("/")[-1])))

    # DBTITLE 4,Write to Managed Delta Table (Unity Catalog)
    (bronze_df.write
     .format("delta")
     .mode("overwrite") # Overwrite mode refreshes the source view for this portfolio setup
     .saveAsTable(target_table_name))
     
    print(f"Successfully wrote data to {target_table_name}. Row count: {bronze_df.count()}")

except Exception as e:
    print(f"CRITICAL ERROR in Ingestion Pipeline: {str(e)}")
    sys.exit(1)